### Explo : sectorial and sub-sectorial indicators - all types of indicators 

In [66]:
import pandas as pd
from pathlib import Path

# ── 1. Chargement de la base GLOBALE (indicators) ──
dir_global = Path("./indicators")
if dir_global.exists():
    files_global = list(dir_global.glob("*.parquet"))
    df_global = pd.concat([pd.read_parquet(f) for f in files_global], ignore_index=True)
    df_global = df_global.groupby('period').max().reset_index()
    df_global['period'] = pd.to_datetime(df_global['period'])
    df_global = df_global.sort_values('period')
    print(f"✓ Base GLOBALE chargée : {len(df_global)} jours, de {df_global['period'].min().date()} à {df_global['period'].max().date()}")
else:
    df_global = None
    print("⚠ Dossier ./indicators introuvable.")

# ── 2. Chargement de la base RÉGIONALE (indicators_geo) ──
dir_geo = Path("./indicators_geo") # Modifiez si votre dossier s'appelle autrement
if dir_geo.exists():
    files_geo = list(dir_geo.glob("*.parquet"))
    df_geo = pd.concat([pd.read_parquet(f) for f in files_geo], ignore_index=True)
    df_geo = df_geo.groupby(['period', 'region_key']).max().reset_index()
    df_geo['period'] = pd.to_datetime(df_geo['period'])
    df_geo = df_geo.sort_values(['period', 'region_key'])
    print(f"✓ Base RÉGIONALE chargée : {len(df_geo)} lignes, avec les régions {list(df_geo['region_key'].unique())}")
else:
    df_geo = None
    print("⚠ Dossier ./indicators_geo introuvable.")

✓ Base GLOBALE chargée : 4123 jours, de 2015-02-18 à 2026-06-19
✓ Base RÉGIONALE chargée : 27872 lignes, avec les régions ['Africa', 'China', 'European Union', 'France', 'India', 'Japan', 'Lebanon', 'Middle East', 'North America', 'Russia', 'South America', 'South East Asia', 'US']


In [67]:
import plotly.express as px

def plot_gdelt_indicator(df_global, df_geo, metric, mode="sectors", sector_name=None, sector_keys=None, region_keys=None, region_filter=None, rolling_window=14):
    """
    Trace la série temporelle d'un indicateur GDELT en piochant intelligemment dans la base globale ou régionale.

    Paramètres:
    - df_global : DataFrame des indicateurs mondiaux.
    - df_geo : DataFrame des indicateurs régionaux.
    - metric : str, l'indicateur à tracer (ex: 'att_weight', 'sent_cont', 'axs_bin').
    - mode : 'sectors', 'categories', ou 'regions'.
    - sector_name : str, requis si mode='categories' ou 'regions'.
    - sector_keys : list, requis si mode='sectors'.
    - region_keys : list, requis si mode='regions' (ex: ['US', 'France']).
    - region_filter : str, optionnel. Permet d'appliquer le mode 'sectors/categories' sur UN seul pays.
    - rolling_window : int, jours pour la moyenne mobile (1 = données brutes).
    """
    
    # ── 1. ROUTEUR AUTOMATIQUE (Choix du DataFrame) ──
    if mode == "regions" or region_filter is not None:
        if df_geo is None or df_geo.empty:
            raise ValueError("Vous demandez une analyse géo, mais 'df_geo' est vide ou non chargé.")
        working_df = df_geo.copy()
    else:
        if df_global is None or df_global.empty:
            raise ValueError("Vous demandez une analyse globale, mais 'df_global' est vide ou non chargé.")
        working_df = df_global.copy()

    title_suffix = ""

    # ── 2. APPLICATION DU FILTRE GÉO (Si hors mode "regions") ──
    if region_filter and mode in ['sectors', 'categories']:
        working_df = working_df[working_df['region_key'] == region_filter]
        title_suffix = f" [Filtre Région : {region_filter}]"

    # ── 3. PRÉPARATION DES DONNÉES SELON LE MODE ──
    if mode == "categories":
        if not sector_name:
            raise ValueError("Spécifiez 'sector_name' (ex: 'energy') en mode 'categories'.")
        prefix = f"{metric}_{sector_name}"
        cols_to_plot = [c for c in working_df.columns if c == prefix or c.startswith(prefix + "_")]
        
        plot_df = working_df.set_index('period')[cols_to_plot]
        title = f"Indicateur : {metric} | Secteur : {sector_name.upper()} & ses catégories"

    elif mode == "sectors":
        if not sector_keys:
            raise ValueError("Fournissez 'sector_keys' (ex: ['energy', 'finance']) en mode 'sectors'.")
        cols_to_plot = [f"{metric}_{s}" for s in sector_keys if f"{metric}_{s}" in working_df.columns]
        
        plot_df = working_df.set_index('period')[cols_to_plot]
        title = f"Indicateur : {metric} | Comparaison inter-secteurs"

    elif mode == "regions":
        if not sector_name:
            raise ValueError("Spécifiez 'sector_name' (ex: 'energy') en mode 'regions'.")
        target_col = f"{metric}_{sector_name}"
        if target_col not in working_df.columns:
            raise ValueError(f"La colonne '{target_col}' est introuvable.")
        
        if region_keys:
            working_df = working_df[working_df['region_key'].isin(region_keys)]
        
        plot_df = working_df.pivot(index='period', columns='region_key', values=target_col)
        title = f"Indicateur : {metric} | Secteur : {sector_name.upper()} | Comparaison Géo"

    else:
        raise ValueError("Le mode doit être 'sectors', 'categories', ou 'regions'.")

    if plot_df.empty or plot_df.columns.empty:
        print(f"⚠ Aucune donnée trouvée. Vérifiez vos clés et métriques.")
        return

    title += title_suffix

    # ── 4. LISSAGE (MOYENNE MOBILE) ──
    if rolling_window > 1:
        plot_df = plot_df.rolling(window=rolling_window, min_periods=1).mean()
        title += f" (Moyenne mobile {rolling_window}j)"

    # ── 5. FORMATAGE PLOTLY ──
    melted = plot_df.reset_index().melt(id_vars=['period'], var_name='Série', value_name='Valeur')
    
    if mode != "regions":
        melted['Série'] = melted['Série'].str.replace(f"{metric}_", "", regex=False)

    # ── 6. AFFICHAGE ──
    fig = px.line(
        melted, x='period', y='Valeur', color='Série', title=title,
        template="plotly_white", render_mode="svg"
    )
    fig.update_xaxes(title_text="Date", rangeslider_visible=True)
    fig.update_yaxes(title_text="Valeur")
    fig.update_layout(legend_title_text="Légende", hovermode="x unified")
    fig.show()

In [68]:
plot_gdelt_indicator(
    df_global, df_geo, 
    metric='att_weight', mode='sectors', sector_keys=['energy', 'finance', 'tech']
)

In [69]:
plot_gdelt_indicator(
    df_global, df_geo, 
    metric='sent_cont_weight', mode='regions', sector_name='energy', region_keys=['US', 'China', 'European Union']
)

In [70]:
plot_gdelt_indicator(
    df_global, df_geo, 
    metric='att_weight', mode='sectors', sector_keys=['energy', 'finance'], region_filter='France'
)

In [72]:
plot_gdelt_indicator(
    df_global, df_geo, 
    metric='att', mode='categories', sector_name = 'energy', region_filter='European Union'
)